In [1]:
import os
os.environ["NPU_VISIBLE_DEVICES"]="7"
os.environ["ASCEND_RT_VISIBLE_DEVICES"]="7"
import json
from typing import Dict, List, Any
from tqdm import tqdm
from functools import partial

import math
import numpy as np
import matplotlib.pyplot as plt

import torch
import torch_npu
from transformers import AutoTokenizer
from datasets import load_dataset, concatenate_datasets

/home/lihz/miniconda3/envs/workspace/lib/python3.10/site-packages/torch_npu/utils/path_manager.py:82: UserWarning: Warning: The /usr/local/Ascend/ascend-toolkit/latest owner does not match the current user.
  warnings.warn(f"Warning: The {path} owner does not match the current user.")
/home/lihz/miniconda3/envs/workspace/lib/python3.10/site-packages/torch_npu/utils/path_manager.py:82: UserWarning: Warning: The /usr/local/Ascend/ascend-toolkit/8.0.RC2/aarch64-linux/ascend_toolkit_install.info owner does not match the current user.
  warnings.warn(f"Warning: The {path} owner does not match the current user.")


In [2]:
base_model = "/data/pretrained-models/meta/Llama-3.2-3B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(base_model)

In [3]:
base_dataset = "/data/datasets/Llama-3.2-3B-Instruct-evals"
general_datasets = [
    "/data/datasets/ultrachat_200k",
]
reason_datasets = [
    "allenai/cosmos_qa",
    "rajpurkar/squad_v2"
]
math_datasets = [
    "gsm8k",
    "/data/datasets/MathInstruct",
    "EleutherAI/hendrycks_math"
]
magicoder_datasets = [
    "/data/datasets/Magicoder-Evol-Instruct-110K",
]
leetcode_datasets = [
    "greengerong/leetcode"
]

In [4]:
def preprocess_gsm8k(examples:Dict[str, Any], tokenizer:AutoTokenizer)->Dict[str, Any]:
    prefix = "Given the following problem, reason and give a final answer to the problem.\nProblem: {{question}}\nYour response should end with \"The final answer is [answer]\" where [answer] is the response to the problem.\n"
    icl = [
        {
            "role" : "user",
            "content" : "There are 15 trees in the grove. Grove workers will plant trees in the grove today. After they are done, there will be 21 trees. How many trees did the grove workers plant today?"
        },
        {
            "role" : "assistant",
            "content" : "There are 15 trees originally. Then there were 21 trees after some more were planted. So there must have been 21 - 15 = 6. The final answer is 6"
        },
        {
            "role": "user",
            "content": "If there are 3 cars in the parking lot and 2 more cars arrive, how many cars are in the parking lot?"
        },
        {
            "role": "assistant",
            "content" : "There are originally 3 cars. 2 more cars arrive. 3 + 2 = 5. The final answer is 5"
        },
        {
            "role": "user",
            "content" : "Leah had 32 chocolates and her sister had 42. If they ate 35, how many pieces do they have left in total?",
        },
        {
            "role" : "assistant",
            "content" : "Originally, Leah had 32 chocolates. Her sister had 42. So in total they had 32 + 42 = 74. After eating 35, they had 74 - 35 = 39. The final answer is 39"
        },
        {
            "role" : "user",
            "content" : "Jason had 20 lollipops. He gave Denny some lollipops. Now Jason has 12 lollipops. How many lollipops did Jason give to Denny?"
        },
        {
            "role" : "assistant",
            "content" : "Jason started with 20 lollipops. Then he had 12 after giving some to Denny. So he gave Denny 20 - 12 = 8. The final answer is 8"
        },
        {
            "role" : "user",
            "content" : "Shawn has five toys. For Christmas, he got two toys each from his mom and dad. How many toys does he have now?"
        },
        {
            "role" : "assistant",
            "content" : "Shawn started with 5 toys. If he got 2 toys each from his mom and dad, then that is 4 more toys. 5 + 4 = 9. The final answer is 9"
        },
        {
            "role" : "user",
            "content" : "There were nine computers in the server room. Five more computers were installed each day, from monday to thursday. How many computers are now in the server room?"
        },
        {
            "role" : "assistant",
            "content" : "There were originally 9 computers. For each of 4 days, 5 more computers were added. So 5 * 4 = 20 computers were added. 9 + 20 is 29. The final answer is 29"
        },
        {
            "role" : "user",
            "content" : "Michael had 58 golf balls. On tuesday, he lost 23 golf balls. On wednesday, he lost 2 more. How many golf balls did he have at the end of wednesday?"
        },
        {
            "role" : "assistant",
            "content" : "Michael started with 58 golf balls. After losing 23 on tuesday, he had 58 - 23 = 35. After losing 2 more, he had 35 - 2 = 33 golf balls. The final answer is 33"
        },
        {
            "role" : "user",
            "content" : "Olivia has $23. She bought five bagels for $3 each. How much money does she have left?"
        },
        {
            "role" : "assistant",
            "content" : "Olivia had 23 dollars. 5 bagels for 3 dollars each will be 5 x 3 = 15 dollars. So she has 23 - 15 dollars left. 23 - 15 is 8. The final answer is 8"
        }
    ]
    for i in range(len(icl)):
        if icl[i]['role'] == "user":
            icl[i]['content'] = prefix.replace("{{question}}", icl[i]['content'])
    return {
        "inst": prefix.replace("{{question}}", examples["question"].strip()),
        "response": examples["answer"].strip().replace("\n", " ").replace("#### ", "The final answer is "),
        "history": icl, 
        "system": ""
        }

def preprocess_math(examples:Dict[str, Any], tokenizer:AutoTokenizer) -> Dict[str, Any]:
    icl = [
        # {
        #     'role': 'user',
        #     'content': """Provide concise and precise solutions to the following math problems. Topics may include algebra, counting and probability, geometry, intermediate algebra, number theory, prealgebra, or precalculus. Show only essential steps and the final answer."""
        # },
        {
            'role': "user",
            'content': "A certain circle's area is $x$ square units, and its circumference is $y$ units. The value of $x + y$ is $80\pi$. What is the radius of the circle, in units?",
        },
        {
            "role": "assistant",
            "content": """If $r$ is the radius of the circle, we know that $x = \pi r^2$ and $y = 2\pi r$. Thus, we have \begin{align*}
x + y &= 80\pi\\
\pi r^2 + 2\pi r &= 80\pi\\
r(r + 2) &= 80.
\end{align*}

We want to find two integers whose product is 80, such that one integer is two more than the other. We note that 80 can be factored as $8\cdot 10$. Therefore, $r = \boxed{8}$.""",
        },
        {
            "role": "user",
            "content": "Consider two positive even integers less than $15$ (not necessarily distinct). When the sum of these two numbers is added to their product, how many different possible values may result?"
        },
        {
            "role": "assistant",
            "content": """Let $p$ and $q$ be the two integers; then $p,q \in \{2,4,6,8,10,12,14\}$, giving $7 \times 7 = 49$ total possible pairs $(p,q)$. The question asks for the number of different values of $pq + p + q$. Notice that by Simon's Favorite Factoring Trick, $$pq + p + q = (p+1)(q+1) - 1,$$so it suffices to find the number of different possible values of $(p+1)(q+1)$. Here, $p+1,q+1 \in \{3,5,7,9,11,13,15\}$.

There are $7$ pairs $(p,q)$ where $p+1$ is equal to $q+1$; by symmetry, half of the $42$ remaining pairs correspond to swapping the values of $p$ and $q$, leaving $42/2 = 21$ pairs $(p,q)$. Since most of the possible values of $p+1$ and $q+1$ are prime factors that do not divide into any of the other numbers, we note that most of the values of $(p+1)(q+1)$ will be distinct. The exception are the numbers divisible by $3$ and $5$: $p+1,q+1 \in \{3,5,9,15\}$; then, if $(p+1,q+1) = (3,15)$ or $(5,9)$, then $(p+1)(q+1) = 45$.

Hence, there are exactly $21-1+7 = \boxed{27}$ distinct possible values of $pq + p + q$."""
        },
        {
            "role": "user",
            "content": "A card is chosen at random from a standard deck of 52 cards, and then it is replaced and another card is chosen. What is the probability that at least one of the cards is a diamond or an ace?"
        },
        {
            "role": "assistant",
            "content": """There are 16 cards in a standard deck which are either diamonds or aces. The probability that neither card chosen is a diamond or an ace is $\left( \frac{36}{52} \right) ^2=\left( \frac{9}{13} \right) ^2=\frac{81}{169}$. Therefore, the probability that at least one of the cards chosen was a diamond or an ace is $1-\frac{81}{169}=\boxed{\frac{88}{169}}$."""
        },
        {
            "role": "user",
            "content": "What is the remainder when $1 + 2 + 3 + 4 + \dots + 9 + 10$ is  divided by 8?"
        },
        {
            "role": "assistant",
            "content": """Notice that we can pair many of these terms: \[1+7=2+6=3+5=8,\]so the remainder we want is the same as the remainder when $4+9+10$ is divided by 8.  We also see that this is the remainder when  \[4+1+2=7\]is divided by 8, so the answer is $\boxed{7}$."""
        },
        {
            "role": "user",
            "content": """The real numbers $a$ and $b$ satisfy
\[\begin{pmatrix} 2 \\ a \\ -7 \end{pmatrix} \times \begin{pmatrix} 5 \\ 4 \\ b \end{pmatrix} = \mathbf{0}.\]Enter the ordered pair $(a,b).$"""
        },
        {
            "role": "assistant",
            "content": """In general, $\mathbf{v} \times \mathbf{w} = \mathbf{0}$ if and only if the vectors $\mathbf{v}$ and $\mathbf{w}$ are proportional.  Thus, the vectors $\begin{pmatrix} 2 \\ a \\ -7 \end{pmatrix}$ and $\begin{pmatrix} 5 \\ 4 \\ b \end{pmatrix}$ are proportional.  Thus,
\[\frac{5}{2} = \frac{4}{a} = \frac{b}{-7}.\]Solving, we find $(a,b) = \boxed{\left( \frac{8}{5}, -\frac{35}{2} \right)}.$"""
        }
    ]
    return {
        'inst': examples['problem'],
        'response': examples['solution'],
        "history": icl,
        "system": """Provide concise and precise solutions to the following math problems. Topics may include algebra, counting and probability, geometry, intermediate algebra, number theory, prealgebra, or precalculus. Show only essential steps and the final answer."""
    }

def preprocess_cosmos(examples: Dict[str, Any], tokenizer:AutoTokenizer) -> Dict[str, Any]:
    prompt = """Given the context and question, select the most appropriate answer from the provided options. Then, repeat the full content of the selected option in your response.

Context:
Good Old War and person L: I saw both of these bands Wednesday night, and they both blew me away. Seriously. Good Old War is acoustic and makes me smile. I really can not help but be happy when I listen to them; I think it’s the fact that they seemed so happy themselves when they played.

Question:
In the future, will this person go to see other bands play?

A. None of the above choices.
B. This person likes music and likes to see the show, they will see other bands play.
C. This person only likes Good Old War and Person L, no other bands.
D. Other Bands is not on tour and this person cannot see them.

Answer:
A. None of the above choices."""
    prefix = "A"
    answer = 0
    if examples['label'] == 1:
        prefix = "B"
        answer = 1
    elif examples['label'] == 2:
        prefix = "C"
        answer = 2
    elif examples['label'] == 3:
        prefix = "D"
        answer = 3
    answer = f"answer{answer}"
    return {
        'inst': f"""Context:
{examples['context'].strip()}

Question:

{examples['question'].strip()}

A. {examples['answer0'].strip()}
B. {examples['answer1'].strip()}
C. {examples['answer2'].strip()}
D. {examples['answer3'].strip()}""",
        'response': f"{prefix}. {examples[answer].strip()}",
        "history": [],
        "system": prompt,
        # 'history': [
        #     {
        #         'role': 'user',
        #         'content': prompt
        #     }
        # ]
    }

def preprocess_sqaure(examples: Dict[str, Any], tokenizer:AutoTokenizer) -> Dict[str, Any]:
    prompt = """Given the context and the question, provide a concise, accurate answer in the format `[answer_start]: [text]`, where `answer_start` is the index of where the answer starts in the context, and `text` is the answer itself."""
    question = f"""Context:

{examples['context'].strip()}

Question:

{examples['question'].strip()}"""

    answers = [
        f"""{answer_start}: {txt.strip()}""" if txt.strip()[-1] == '.'
        else f"""{answer_start}: {txt.strip()}."""
        for txt, answer_start in zip(examples['answers']['text'], examples['answers']['answer_start'])
    ]
    if len(answers) == 0:
        answers_ = "-1: No answer founded."
    else:
        answers_ = answers[0]
        for item in answers[1:]:
            answers += f"\n{item}"
    return {
        "inst": question,
        "response": answers_,
        "history": [
            # {
            #     "role": "user",
            #     "content": prompt
            # },
            {
                "role": "user",
                "content": """Context:

The most widely spoken family of languages in southern Europe are the Romance languages, the heirs of Latin, which have spread from the Italian peninsula, and are emblematic of Southwestern Europe. (See the Latin Arch.) By far the most common romance languages in Southern Europe are: Italian, which is spoken by over 50 million people in Italy, San Marino, and the Vatican; and Spanish, which is spoken by over 40 million people in Spain and Gibraltar. Other common romance languages include: Romanian, which is spoken in Romania and Moldova; Portuguese, which is spoken in Portugal; Catalan, which is spoken in eastern Spain; and Galician, which is spoken in northwestern Spain.

Question:

What are the three main areas of southern Europe where Italian speakers can be found?"""
            },
            {
                "role": "assistant",
"content": """339: Italy, San Marino, and the Vatican."""
            }
        ],
        "system": prompt,
    }

def preprocess_llama3_eval(examples:Dict[str, Any], tokenizer:AutoTokenizer) -> Dict[str, Any]:
    input_final_prompts = examples['input_final_prompts'][0].replace("<|eot_id|>", "")
    input_final_prompts = input_final_prompts.replace("<|start_header_id|>user<|end_header_id|>", "#$%2$%$#")
    input_final_prompts = input_final_prompts.split("#$%2$%$#")[1:]
    input_final_prompts = [item.split("<|start_header_id|>assistant<|end_header_id|>") for item in input_final_prompts]
    history = []
    for question, answer in input_final_prompts[-1]:
        history.append({"role": "user", "content": question})
        history.append({"role": "assistant", "content": answer})
    return {
        "inst": input_final_prompts[-1][0],
        "response": examples['output_prediction_text'][0],
        "history": history,
        "system": ""
    }

def preprocess_ultrachat(examples:Dict[str, Any], tokenizer:AutoTokenizer)->Dict[str,Any]:
    messages = examples['messages']

    return { 
        "inst": messages[-2]['content'],
        "response": messages[-1]['content'],
        "history": messages[:-2] if len(messages) > 2 else [],
        "system": "",
    }

In [5]:
def task_preprocess(example:Dict[str, str], tokenizer:AutoTokenizer, task:str="humaneval")->Dict[str, str]:
    if task == "humaneval":
        instruction_prefix = "Please provide a self-contained Python script that solves the following problem in a markdown code block:"
        response_prefix = "Below is a Python script with a self-contained function that solves the problem and passes corresponding tests:"
        # some random words which servcleaes as the splitter
        _MAGIC_SPLITTER_ = "-[[]]-this-is-really-our-highest-priority-[[]]-"
        task_prompt = f"""\
{instruction_prefix}
```
{example['prompt'].strip()}
```
"""
        response = f"""\
{response_prefix}
```python
{example}
```
"""
        task_prompt = tokenizer.apply_chat_template(
            [
                {"role": "user", "content": task_prompt},
                {"role": "assistant", "content": response},
            ],
            tokenize=False,
        ).split(_MAGIC_SPLITTER_)[0]
        return {
            "inst": task_prompt,
        }
    elif task == "mbpp":
        instruction_prefix = "Please provide a self-contained Python script that solves the following problem in a markdown code block:"
        response_prefix = "Below is a Python script with a self-contained function that solves the problem and passes corresponding tests:"
        # some random words which servcleaes as the splitter
        _MAGIC_SPLITTER_ = "-[[]]-this-is-really-our-highest-priority-[[]]-"
        python_prefix = 'Write a python function to '
        func_prefix = 'Write a function to '
        if python_prefix in example['prompt']:
            prefix = python_prefix
        elif func_prefix in example['prompt']:
            prefix = func_prefix
        else:
            prefix = ""
        prompt = example['prompt'].replace(prefix, '').strip().capitalize()
        task_prompt = f"""\
{instruction_prefix}
```
{example['code'].split(":")[0].strip()}:
    \"\"\"
    {prompt}
    >>> {example['test_list'][0].replace("assert", "").strip()}
    True
    \"\"\"
```
"""
        response = f"""\
{response_prefix}
```python
{_MAGIC_SPLITTER_}
```
"""
        task_prompt = tokenizer.apply_chat_template(
            [
                {"role": "user", "content": task_prompt},
                {"role": "assistant", "content": response},
            ],
            tokenize=False,
        ).split(_MAGIC_SPLITTER_)[0]
        return {
            "inst": task_prompt,
        }
    elif task == "magicoder":
        return {
            "inst": example['instruction'].strip(),
            "response": example['response'].strip(),
            "history": [],
            "system": "",
        }
    elif task == "mathinstruct":
        return {
            "inst": example['instruction'].strip(),
            "response": example['output'].strip(),
            "history": [],
            "system": "",
        }
    elif task == "leetcode":
        content = [item.strip() for item in example['content']]
        java = [item.strip() for item in example['java']]
        python = [item.strip() for item in example['python']]
        cpp = [item.strip() for item in example['c++']]
        javascript = [item.strip() for item in example['javascript']]
        # for c, j, p, cpp, js in zip(content, java, python, java, cpp, javascript):
        #     leetcode.extend([c, j, p, cpp, js])
        inst, response, history, system = [], [], [], []
        for item in zip(content, java, python, cpp, javascript):
            inst.extend([item[0]] * len(item[1:]))
            response.extend(item[1:])
            history.extend([[] for _ in range(len(item[1:]))])
            system.extend([""] * len(item[1:]))
        return {
            "inst": inst,
            "response": response,
            "history": history,
            "system": system
        }

In [6]:
general_datasets = [
    load_dataset(
        general_datasets[0],
        num_proc=8,
    )['train_sft'],
]
general_datasets = [
    general_datasets[0].filter(
        lambda x: len(x['messages']) > 1,
    ).map(
        partial(preprocess_ultrachat, tokenizer=tokenizer),
        num_proc=8
    )
]
reason_datasets = [
    load_dataset(
        reason_datasets[0],
        num_proc=8,
    )['train'].map(
        partial(preprocess_cosmos, tokenizer=tokenizer),
        num_proc=8,
    ),
    load_dataset(
        reason_datasets[1],
        num_proc=8,
    )['train'].map(
        partial(preprocess_sqaure, tokenizer=tokenizer),
        num_proc=8,
    ),
]

math_config = [
    'algebra',
    'counting_and_probability',
    'geometry',
    'intermediate_algebra',
    'number_theory',
    'prealgebra',
    'precalculus'
]

math_datasets = [
    load_dataset(
        math_datasets[0],
        "main",
        num_proc=8,
    )['train'].map(
        partial(preprocess_gsm8k, tokenizer=tokenizer),
        num_proc=8,
    ),
    load_dataset(
        math_datasets[1],
        num_proc=8,
    ).map(
        partial(task_preprocess, tokenizer=tokenizer, task="mathinstruct"),
        num_proc=8,
    )['train'],
    concatenate_datasets([
        load_dataset(
            math_datasets[2],
            item,
            num_proc=8
        )['train'] for item in math_config
    ]).map(
        partial(preprocess_math, tokenizer=tokenizer),
        num_proc=8,
    )
]

magicoder_datasets = [
    load_dataset(
        item,
        num_proc=8,
    ).map(
        partial(task_preprocess, tokenizer=tokenizer, task="magicoder"),
        num_proc=8,
    )['train'] for item in magicoder_datasets
]

leetcode_datasets = [
    load_dataset(
        item,
        num_proc=8,
    )['train'] for item in leetcode_datasets
]
leetcode_datasets = [
    item.map(
        partial(task_preprocess, tokenizer=tokenizer, task="leetcode"),
        num_proc=8,
        batched=True,
        remove_columns=item.column_names
    )
    for item in leetcode_datasets
]

Using the latest cached version of the module from /data/lihz/.cache/huggingface/modules/datasets_modules/datasets/allenai--cosmos_qa/3e18538cbfdb2c04189b16642715f0f6da3e97ed5df0aadcec3641245b2cf157 (last modified on Sun Dec 22 17:36:47 2024) since it couldn't be found locally at allenai/cosmos_qa, or remotely on the Hugging Face Hub.


In [7]:
# print(magicoder_datasets[0][4]['inst'])
# print(mbpp_datasets[0][5]['inst'])
# print(mbpp_datasets[0][5]['code'])

In [8]:
# print(humaneval_datasets[0][4]['inst'])
# print(humaneval_datasets[0][1]['canonical_solution'])

In [9]:
print(leetcode_datasets[0])

Dataset({
    features: ['inst', 'response', 'history', 'system'],
    num_rows: 9440
})


In [10]:
# print(general_datasets[2][0]['input_final_prompts'][0])
print(general_datasets[0][200]['inst'])
# print(general_datasets[0][200]['history'][-1])

That was a great horror story! Can you add more foreshadowing and suspense to the beginning to make the ending even more shocking?


In [11]:
# print(math_datasets[0][0]['input_final_prompts'][0])
# print(math_datasets[0][0]['input_correct_responses'])
# print(math_datasets[-1][0]['input_final_prompts'][0])
# print(math_datasets[-1][0]['history'][0][0])
print(math_datasets[-1][1065]['inst'])

Find all solutions $x$ of the inequality $$\frac{5}{24} + \left|x-\frac{11}{48}\right| < \frac{5}{16}.$$Express your answer in interval notation, simplifying all fractions in your answer.


In [12]:
print(math_datasets[1][1]['inst'])
print(math_datasets[1][1]['output'])

How many ways can the letters in the word COMMON be arranged?
Answer Choices: (A) 6 (B) 30 (C) 90 (D) 120 (E) 180
Let's solve the multi-choice question step by step.
According to the above the # of permutations of 6 letters COMMON out of which 2 O's and 2 M's are identical is 6!2!∗2!=180
The answer is E.


In [13]:
# print(magicoder_datasets[0][3]['inst'])
# print(magicoder_datasets[0][3].keys())
print(magicoder_datasets[0][3]['response'])

This task requires writing of a significant volume of code, which is not fully suitable for a text-based medium. However, I will outline a general solution using Python and scikit-learn. We'll use "CountVectorizer" for bag-of-words model and "TfidVectorizer" for TF-IDF. To handle different languages, we can use 'langdetect' library.

1. Import required libraries
```python
import pandas as pd
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import classification_report, accuracy_score, confusion_matrix
from langdetect import detect
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from nltk.tokenize import word_tokenize
import nltk
nltk.download('punkt')
nltk.download('wordnet')
nltk.download('stopwords')
```

2. Load sentence data and labels. For example, if data is stored in a csv format:
```python
data = pd.read_cs

In [14]:
print(len(general_datasets[0]),
      len(reason_datasets[0]),
      len(reason_datasets[1]),
      len(math_datasets[0]), 
      len(math_datasets[1]),
      len(math_datasets[2]),
      len(leetcode_datasets[0]), 
      len(magicoder_datasets[0]))

207865 25262 130319 7473 262039 7500 9440 111183


In [15]:
mix_domains = []

samples = 4000

repeat = 1 if len(general_datasets[0]) >= samples else 2
for _ in range(repeat):
    for i, item in enumerate(general_datasets[0]):
        if i == samples + 400:
            break
        mix_domains.append({
            'inputs': item['inst'],
            'task': 'ultrachat',
            'task_label': 0,
            'label': 0,
            'outputs': item['response'],
            'history': item['history'],
            "system": item['system'],
        })
print(len(mix_domains))

4400


In [16]:
print(len(reason_datasets[0]), len(reason_datasets[1]))

25262 130319


In [17]:
repeat = 1 if len(reason_datasets[0]) >= samples else 2
for _ in range(repeat):
    for i, item in enumerate(reason_datasets[0]):
        if i == samples:
            break
        mix_domains.append({
            'inputs': item['inst'],
            'task': 'cosmos',
            'task_label': 1,
            'label': 1,
            'outputs': item['response'],
            'history': item['history'],
            "system": item['system'],
        })
repeat = 1 if len(reason_datasets[1]) >= samples else 2
for _ in range(repeat):
    for i, item in enumerate(reason_datasets[1]):
        if i == samples:
            break
        mix_domains.append({
            'inputs': item['inst'],
            'task': 'square',
            'task_label': 2,
            'label': 1,
            'outputs': item['response'],
            'history': item['history'],
            "system": item['system'],
        })

In [18]:
repeat = 1 if len(math_datasets[0]) >= samples else 2
for _ in range(repeat):
    for i, item in enumerate(math_datasets[0]):
        if i == samples:
            break
        mix_domains.append({
            'inputs': item['inst'],
            'task': 'gsm8k',
            'task_label': 3,
            'label': 2,
            'outputs': item['response'],
            'history': item['history'],
            "system": item['system'],
        })
print(len(mix_domains))

repeat = 1 if len(math_datasets[1]) >= samples else 2
for _ in range(repeat):
    for i, item in enumerate(math_datasets[1]):
        if i == samples:
            break
        mix_domains.append({
            'inputs': item['inst'],
            'task': 'mathinstruct',
            'task_label': 4,
            'label': 2,
            'outputs': item['response'],
            'history': item['history'],
            "system": item['system'],
        })
print(len(mix_domains))

repeat = 1 if len(math_datasets[2]) >= samples else 2
for _ in range(repeat):
    for i, item in enumerate(math_datasets[2]):
        if i == samples:
            break
        mix_domains.append({
            'inputs': item['inst'],
            'task': 'math',
            'task_label': 5,
            'label': 2,
            'outputs': item['response'],
            'history': item['history'],
            "system": item['system'],
        })
print(len(mix_domains))

16400
20400
24400


In [19]:

repeat = 1 if len(leetcode_datasets[0]) >= samples else 2
for _ in range(repeat):
    for i, item in enumerate(leetcode_datasets[0]):
        if i == samples:
            break
        mix_domains.append({
            'inputs': item['inst'],
            'task': 'leetcode',
            'task_label': 6,
            'label': 3,
            'outputs': item['response'],
            'history': item['history'],
            "system": item['system'],
        })
print(len(mix_domains))

repeat = 1 if len(magicoder_datasets[0]) >= samples else 2
for _ in range(repeat):
    for i, item in enumerate(magicoder_datasets[0]):
        if i == samples + 500:
            break
        mix_domains.append({
            'inputs': item['inst'],
            'task': 'magicoder',
            'task_label': 7,
            'label': 3,
            'outputs': item['response'],
            'history': item['history'],
            "system": item['system'],
        })
print(len(mix_domains))

28400
32900


In [20]:
print(len(mix_domains))

32900


In [21]:
# with open("/data/lihz/datasets/mix_domains/mix_domains_48_v5.jsonl", 'w') as f:
#     for item in mix_domains:
#         f.write(json.dumps(item) + '\n')

In [22]:
import json
with open("/data/lihz/datasets/mix_domains/mix_domains_48_v5.jsonl", 'r') as f:
    for i, line in enumerate(f):

        if json.loads(line)['history'] is None:
            print(i)
            break
        if json.loads(line)['history'] is not None:
            if len(json.loads(line)['history']) > 0:

                for item in json.loads(line)['history']:
                    if len(item) < 2:
                        print(item)
                        print(json.loads(line)['inputs'])
                        print(i)
                        assert False

# with open("/data/lihz/datasets/mix_domains_eval/mix_domains_eval_v10.jsonl", 'r') as f:
#     for i, line in enumerate(f):
#         if json.loads(line)['history'] is None:
#             print(i)
#             break

In [23]:
with open("/data/lihz/datasets/mix_domains_eval/mix_domains_eval_x1.jsonl", 'r') as f:
    for i, line in enumerate(f):
        if json.loads(line)['history'] is None:
            print(i)
            break

In [24]:
from datasets import load_dataset
json_dataset = load_dataset(
    "json",
    data_files=["/data/lihz/datasets/mix_domains/mix_domains_48_v5.jsonl"],
)
# json_dataset = load_dataset(
#     "json",
#     data_files=["/data/lihz/datasets/mix_domains_eval/mix_domains_eval_x1.jsonl"],
# )

Generating train split: 0 examples [00:00, ? examples/s]

In [25]:
print(len(json_dataset['train']))

32900


In [26]:
# import os
# del os.environ['HF_ENDPOINT']
# os.environ['http_proxy']="127.0.0.1:9150"
# os.environ['https_proxy']="127.0.0.1:9150"
# from datasets import load_dataset
# math_dataset = load_dataset("EleutherAI/hendrycks_math", 'precalculus')
# # print(os.environ['http_proxy'])
# # 'algebra', 'counting_and_probability', 'geometry', 'intermediate_algebra', 'number_theory', 'prealgebra', 'precalculus'
# print(type(math_dataset), math_dataset.keys(), len(math_dataset['train']), len(math_dataset['test']))
# print(math_dataset['train'].column_names)
# print(math_dataset['train'][-1]['problem'])
# print(math_dataset['train'][-1]['solution'])

In [27]:
# math_dataset = load_dataset("meta-llama/Llama-3.2-3B-Instruct-evals", 'Llama-3.2-3B-Instruct-evals__math__details')
# # print(len(math_dataset['latest'][0]['input_final_prompts'][0].split("<|start_header_id|>user<|end_header_id|>")[1:]))
# # print(math_dataset['latest'][0]['input_final_prompts'][0].split("<|start_header_id|>user<|end_header_id|>")[1:][0])
# # print(math_dataset['latest'][1002]['input_final_prompts'][0])


In [28]:
# cosmos_dataset = load_dataset("allenai/cosmos_qa")
# print(type(cosmos_dataset), cosmos_dataset.keys(), len(cosmos_dataset['train']))
# print(cosmos_dataset['train'].column_names)
# print(cosmos_dataset['train'][0]['context'])
# print(cosmos_dataset['train'][0]['question'])
# print(cosmos_dataset['train'][0]['answer0'])
# print(cosmos_dataset['train'][0]['answer1'])
# print(cosmos_dataset['train'][0]['answer2'])
# print(cosmos_dataset['train'][0]['answer3'])
# print(cosmos_dataset['train'][0]['label'], type(cosmos_dataset['train'][0]['label']))

In [29]:
# square_dataset = load_dataset("rajpurkar/squad_v2")
# print(type(square_dataset), square_dataset.keys(), len(square_dataset['train']))
# print(square_dataset['train'].column_names)
# tag = 11000
# print(square_dataset['train'][tag]['context'])
# print(square_dataset['train'][tag]['question'])
# print(square_dataset['train'][tag]['answers'])

In [30]:
# from datasets import load_dataset

# gsm8k_dataset = load_dataset("gsm8k", "main")
# print(gsm8k_dataset['train'].column_names)
# print(gsm8k_dataset['train'][0]['question'])
# print(gsm8k_dataset['train'][0]['answer'])